# 03 — Treino rápido: LIBERO com frames pré-processados (Drive)

Versão enxuta do `01_treino_libero.ipynb` para quando o dataset **já foi
pré-processado** (ver `scripts/preprocess_dataset.py`) e o resultado está
salvo como `.tar.gz` no seu Google Drive.

O que este notebook faz, na ordem:
1. Clona o repo (branch `correcoes-set2026`) e instala as dependências
2. Monta o Drive e extrai o `.tar.gz` pro disco LOCAL do Colab (`/content`)
3. Ajusta `device_index` do config pro Colab (o valor salvo é para a
   máquina do CEPEDI, que tem 2 GPUs -- no Colab normalmente há só 1)
4. Chama `scripts/train.py --preprocessed-dir ... --resume`, com log em
   arquivo (sobrevive mesmo se a célula for interrompida)
5. Plota as curvas de treino/validação

Diferente do `01`, este notebook **não reimplementa** o loop de treino --
ele chama o `train.py` da raiz do repo, que já tem todas as correções
(seed global, resume com scaler, top-k real, annealing, etc). Isso evita
duplicar lógica entre o notebook e o script de linha de comando.

**Pré-requisito:** você já rodou o pré-processamento (Colab ou local) e
tem um arquivo `preprocessed_libero40.tar.gz` em `MyDrive/`. Se não tiver
ainda, use o notebook/células de pré-processamento primeiro.

## 1. Repositório e ambiente

In [ ]:
!git clone -b correcoes-set2026 https://github.com/rafaelheydt/act-lang.git
%cd act-lang

!pip install -q -e ".[language]" "lerobot[libero]" 

## 2. Drive: extrair o pré-processado

Extrai para o disco **local** do Colab (`/content`), não para o Drive --
leitura de centenas de milhares de arquivos pequenos direto do Drive é
lenta; o `.tar.gz` existe exatamente para evitar isso.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

TAR_PATH = "/content/drive/MyDrive/preprocessed_libero40.tar.gz"  # ajuste se o nome/local for outro
PREPROCESSED_DIR = "/content/preprocessed_libero40"

!tar -xzf {TAR_PATH} -C /content
print("extraído em", PREPROCESSED_DIR)

## 3. Config: ajustar `device_index` para o Colab

`configs/libero_40tasks_language.py` tem `device_index: 1`, escolhido para
a máquina do CEPEDI (2 GPUs -- 0=A2000, 1=RTX 3050). No Colab normalmente
só existe 1 GPU (índice 0); deixamos `None` para o `pick_device` escolher
sozinho, funcionando em qualquer ambiente com 1+ GPUs.

In [ ]:
!sed -i 's/"device_index": 1,/"device_index": None,  # None = auto -- era 1 p\/ CEPEDI (2 GPUs)/' configs/libero_40tasks_language.py
!grep -n "device_index" configs/libero_40tasks_language.py

## 4. Treino

Chama `train.py` como subprocesso, com log salvo em arquivo (`-u` desliga
o buffer -- sem isso, uma desconexão no meio perde as últimas linhas que
ainda não tinham sido escritas). `--resume` retoma de
`last_checkpoint.pt` se ele já existir no diretório de checkpoints (Drive).

Escolha o mecanismo trocando `CONFIG` abaixo: `language_40_film` |
`language_40_token` | `language_40_cross_attn` (os nomes exatos dependem
do que estiver registrado em `CONFIG_REGISTRY`, em `scripts/train.py`).

In [ ]:
CONFIG = "language_40_film"

import subprocess, sys
from datetime import datetime

Path = __import__("pathlib").Path
Path("logs").mkdir(exist_ok=True)
log_path = f"logs/treino_{datetime.now():%Y%m%d_%H%M}.log"

cmd = [
    sys.executable, "-u", "scripts/train.py",
    "--config", CONFIG,
    "--preprocessed-dir", PREPROCESSED_DIR,
    "--resume",
]
print("rodando:", " ".join(cmd))
print("log em:", log_path, "\n" + "=" * 60)

with open(log_path, "a") as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

print("\n" + "=" * 60)
print(f"processo encerrado com código {proc.returncode}"
      + (" -- verifique o log acima" if proc.returncode != 0 else " -- ok"))

## 5. Curvas

Carrega o `history` salvo no `last_checkpoint.pt` (ele viaja junto com o
checkpoint -- não precisa recomputar nada).

In [ ]:
import torch
from act_lang.training.checkpoints import load_checkpoint
from act_lang.utils.runtime import get_checkpoint_dir
from configs.libero_40tasks_language import CONFIG_FILM as cfg  # troque se usou outro mecanismo
from act_lang.models.act import ACT
from act_lang.models.fusion.factory import build_fusion

model = ACT(
    action_dim=cfg["action_dim"], state_dim=cfg["state_dim"], d_model=cfg["d_model"],
    latent_dim=cfg["latent_dim"], chunk_size=cfg["chunk_size"], n_cameras=cfg["n_cameras"],
    n_encoder_layers=cfg["n_encoder_layers"], n_decoder_layers=cfg["n_decoder_layers"],
    n_heads=cfg["n_heads"], dropout=cfg["dropout"], decoder_style=cfg["decoder_style"],
    fusion=build_fusion(cfg["fusion_type"], cfg["d_model"]),
)
checkpoint_dir = get_checkpoint_dir(cfg["experiment_name"])
_, history = load_checkpoint(checkpoint_dir / "last_checkpoint.pt", model, device="cpu")

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss total (recon + kl_weight*KL)")

axes[1].plot(history["train_recon"], label="train (z~q)")
axes[1].plot(history["val_recon"], label="val (z=mu)")
axes[1].plot(history["val_recon_z0"], label="val (z=0)", linestyle="--")
axes[1].set_title("Recon L1 — z0 é a métrica de seleção")

axes[2].plot(history["train_kld"], label="train")
axes[2].plot(history["val_kld"], label="val")
axes[2].set_title("kld_raw")

axes[3].plot(history["train_mu_abs_mean"], label="train")
axes[3].plot(history["val_mu_abs_mean"], label="val")
axes[3].axhline(0, color="gray", linestyle=":", linewidth=1)
axes[3].set_title("|mu| médio — perto de 0 = posterior colapsado")

for ax in axes:
    ax.set_xlabel("época"); ax.legend()
plt.tight_layout(); plt.show()